# Auditoria privada AOT — Etapa 90-B

Notebook somente-leitura para conferir os 1.916 objetos já existentes (MAIN 1.707, SDK 136, subsdk0 73). Não extrai ZIPs, não recompila, não liga e não envia arquivos comerciais. O único arquivo gravado no Drive é o relatório JSON de metadados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

expected_name = 'Kazuma_Traducoes_AOT_Privado'
private_root = Path('/content/drive/MyDrive') / expected_name
if not private_root.is_dir():
    candidates = [p for base in (Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')) if base.exists() for p in base.rglob(expected_name) if p.is_dir()]
    if len(candidates) != 1:
        print('Pastas encontradas:')
        for p in candidates:
            print(p)
        raise RuntimeError(f'Esperava encontrar exatamente uma pasta {expected_name}; ajuste private_root manualmente se necessário.')
    private_root = candidates[0]
print('Pasta privada localizada:', private_root)
print('Nenhum arquivo será extraído, movido ou alterado.')


In [ ]:
import subprocess
from pathlib import Path

repo_dir = Path('/content/Castlevania-PS4-Port')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'kazuma-ps4-radv-strict-link-20261109',
        'https://github.com/hansifelix/Castlevania-PS4-Port.git', str(repo_dir)
    ], check=True)
audit_script = repo_dir / 'tools/etapa90/audit_private_aot_objects.py'
report_path = Path('/content/drive/MyDrive/RELATORIO_ETAPA90B.json')
if report_path.resolve().is_relative_to(private_root.resolve()):
    raise RuntimeError('O relatório precisa ficar fora da pasta privada auditada.')
result = subprocess.run([
    'python3', str(audit_script),
    '--root', str(private_root),
    '--report', str(report_path)
], text=True, capture_output=True, check=False)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
print('Código de saída:', result.returncode)
print('Relatório JSON:', report_path)


## Como interpretar

- `PASS_STRUCTURE_ELF_AND_MANIFESTS`: estrutura, ELF e hashes dos índices únicos conferidos.
- `PASS_STRUCTURE_ELF_MANIFESTS_INCOMPLETE`: ELF/índices passaram, mas manifests não cobrem tudo ou usam formato não reconhecido.
- `FAIL_OR_INCOMPLETE`: lacunas, conflitos, ELF inválido, ZIP ilegível ou outro problema.

**Não compartilhe arquivos `code_*.o`, `table.cpp` ou fontes AOT.** Para a próxima revisão, compartilhe somente `RELATORIO_ETAPA90B.json`.